<a href="https://colab.research.google.com/github/kalyaninivant/Digital_Image_Processing-/blob/main/Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt


# =========================================================
# LOAD IMAGE
# =========================================================
img_path = ""
img = cv2.imread(img_path)


# =========================================================
# 1. BOX FILTER (MEAN FILTER)
# =========================================================
print("Running Box Filter...")

box_3x3 = np.ones((3, 3)) / 9
box_5x5 = np.ones((5, 5)) / 25
box_7x7 = np.ones((7, 7)) / 49

img_3x3 = cv2.filter2D(img, -1, box_3x3)
img_5x5 = cv2.filter2D(img, -1, box_5x5)
img_7x7 = cv2.filter2D(img, -1, box_7x7)

images = [img, img_3x3, img_5x5, img_7x7]
titles = ["Original", "3x3", "5x5", "7x7"]

plt.figure(figsize=(10,5))
for i in range(4):
    plt.subplot(1,4,i+1)
    plt.imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
    plt.title(titles[i])
    plt.axis('off')
plt.tight_layout()
plt.show()


# =========================================================
# 2. WEIGHTED AVERAGE FILTER
# =========================================================
print("Running Weighted Average Filter...")

weighted_filter = np.array([
    [1,2,1],
    [2,4,2],
    [1,2,1]
]) / 16

weighted_img = cv2.filter2D(img, -1, weighted_filter)

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(cv2.cvtColor(weighted_img, cv2.COLOR_BGR2RGB))
plt.title("Weighted Avg")
plt.axis('off')

plt.tight_layout()
plt.show()


# =========================================================
# 3. MEDIAN FILTER
# =========================================================
print("Running Median Filter...")

median_img = cv2.medianBlur(img, 5)

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(cv2.cvtColor(median_img, cv2.COLOR_BGR2RGB))
plt.title("Median (5x5)")
plt.axis('off')

plt.tight_layout()
plt.show()


# =========================================================
# 4. ADD NOISE (SALT & PEPPER + GAUSSIAN)
# =========================================================
print("Adding Noise...")

def add_salt_pepper_noise(image, amount=0.05):
    noisy = image.copy()
    num_salt = int(amount * image.size * 0.5)
    num_pepper = int(amount * image.size * 0.5)

    # Salt
    coords = [np.random.randint(0, i-1, num_salt) for i in image.shape]
    noisy[tuple(coords)] = 255

    # Pepper
    coords = [np.random.randint(0, i-1, num_pepper) for i in image.shape]
    noisy[tuple(coords)] = 0

    return noisy


def add_gaussian_noise(image, mean=0, var=0.01):
    sigma = var ** 0.5
    gauss = np.random.normal(mean, sigma, image.shape)
    noisy = image + gauss * 255
    return np.clip(noisy, 0, 255).astype(np.uint8)


salt_pepper_img = add_salt_pepper_noise(img)
gaussian_img = add_gaussian_noise(img)

plt.figure(figsize=(12,4))
titles = ["Original", "Salt & Pepper", "Gaussian"]
images = [img, salt_pepper_img, gaussian_img]

for i in range(3):
    plt.subplot(1,3,i+1)
    plt.imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
    plt.title(titles[i])
    plt.axis('off')

plt.tight_layout()
plt.show()


# =========================================================
# 5. MIN, MAX, MEDIAN FILTER (CUSTOM)
# =========================================================
print("Applying Min, Max, Median Filters...")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def min_filter(image, k):
    pad = k // 2
    padded = np.pad(image, pad, mode='constant', constant_values=255)
    output = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+k, j:j+k]
            output[i,j] = np.min(region)

    return output


def max_filter(image, k):
    pad = k // 2
    padded = np.pad(image, pad, mode='constant', constant_values=0)
    output = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+k, j:j+k]
            output[i,j] = np.max(region)

    return output


# Apply filters on Gaussian noisy image
gray_gaussian = cv2.cvtColor(gaussian_img, cv2.COLOR_BGR2GRAY)

min_img = min_filter(gray_gaussian, 3)
max_img = max_filter(gray_gaussian, 3)
median_custom = cv2.medianBlur(gray_gaussian, 5)

plt.figure(figsize=(12,4))
titles = ["Gaussian Noise", "Min", "Max", "Median"]
images = [gray_gaussian, min_img, max_img, median_custom]

for i in range(4):
    plt.subplot(1,4,i+1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.axis('off')

plt.tight_layout()
plt.show()


# =========================================================
# 6. IMPULSE NOISE + FILTER COMPARISON
# =========================================================
print("Impulse Noise + Filtering...")

def add_impulse_noise(image, prob=0.05):
    noisy = image.copy()
    h, w = image.shape

    for i in range(h):
        for j in range(w):
            r = np.random.rand()
            if r < prob/2:
                noisy[i,j] = 0
            elif r < prob:
                noisy[i,j] = 255

    return noisy


noisy_img = add_impulse_noise(gray)

min_img = min_filter(noisy_img, 3)
max_img = max_filter(noisy_img, 3)
median_img = cv2.medianBlur(noisy_img, 5)

plt.figure(figsize=(12,4))
titles = ["Noisy", "Min", "Max", "Median"]
images = [noisy_img, min_img, max_img, median_img]

for i in range(4):
    plt.subplot(1,4,i+1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.axis('off')

plt.tight_layout()
plt.show()